# CSE Policy Impact Predictor — University Research Platform
## Notebook 02: Historical Market Visualization and Policy Event Overlays

**Project Title:** Predicting Stock Market Movements Based on Government Policy Events Using Machine Learning  
**Module:** Government Policy Stock Market Prediction  
**Phase:** Phase 2 — Professional Stock Market Timeline Dashboard  

### Objective
This notebook visualizes the Colombo Stock Exchange (CSE) All Share Price Index (ASPI) and S&P SL20 historical time series across the 2020–2025 primary research timeline. It overlays 70 sovereign policy events directly on the index price series, generates event-study return distributions, and produces publication-quality charts for the university thesis.

### 1. Environment and Package Setup
Import data analysis and financial visualization libraries.

In [ ]:
# Auto-install packages in Google Colab
!pip install -q pandas numpy matplotlib seaborn

import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

print(f"Python Version: {sys.version.split()[0]}")
print(f"Pandas Version: {pd.__version__}")

### 2. Path Configuration
Resolve path to processed data generated in Phase 1 (`cleaned_stock_data.csv` and `aligned_policy_stock_data.csv`).

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/research-dev'
except Exception:
    BASE_DIR = os.getcwd()

STOCK_FILE = os.path.join(BASE_DIR, 'data', 'processed', 'cleaned_stock_data.csv')
ALIGNED_FILE = os.path.join(BASE_DIR, 'data', 'processed', 'aligned_policy_stock_data.csv')
REPORTS_DIR = os.path.join(BASE_DIR, 'reports', 'phases', 'phase_02')
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"Stock file exists: {os.path.exists(STOCK_FILE)}")
print(f"Aligned file exists: {os.path.exists(ALIGNED_FILE)}")

### 3. Load Processed Series and Filter 2020–2025 Focus Timeline
Load 9,772 historical trading days and filter to the 1,400 sessions comprising the 2020–2025 evaluation timeframe.

In [ ]:
df_stock = pd.read_csv(STOCK_FILE)
df_stock['Date_dt'] = pd.to_datetime(df_stock['Date'])
df_stock = df_stock.sort_values('Date_dt').reset_index(drop=True)

# Calculate Technical Moving Averages
df_stock['MA_20'] = df_stock['ASPI'].rolling(20).mean()
df_stock['MA_50'] = df_stock['ASPI'].rolling(50).mean()

# Focus window
df_focus = df_stock[(df_stock['Date_dt'] >= '2020-01-01') & (df_stock['Date_dt'] <= '2025-12-31')].copy()
print(f"Total 2020–2025 sessions: {len(df_focus):,}")
print(f"Starting ASPI close (Jan 2020): {df_focus['ASPI'].iloc[0]:.2f}")
print(f"Ending ASPI close (Dec 2025): {df_focus['ASPI'].iloc[-1]:.2f}")
print(f"All-time period high: {df_focus['ASPI'].max():.2f}")
print(f"All-time period low: {df_focus['ASPI'].min():.2f}")

### 4. Load Aligned Government Policies for the Focus Period
Extract policies aligned to trading days within 2020–2025.

In [ ]:
df_aligned = pd.read_csv(ALIGNED_FILE)
df_aligned['Aligned_Date_dt'] = pd.to_datetime(df_aligned['Aligned_Trading_Date'])

policies_focus = df_aligned[(df_aligned['Aligned_Date_dt'] >= '2020-01-01') & (df_aligned['Aligned_Date_dt'] <= '2025-12-31')].copy()
print(f"Policy events within 2020–2025: {len(policies_focus)}")
print("Breakdown by category:")
print(policies_focus['Policy_Type'].value_counts())

### 5. High-Resolution Figure 1: Historical Stock Timeline with Policy Markers (2020–2025)
Plot the ASPI price curve, moving averages, turnover volume, and color-coded policy markers directly on the trajectory.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), dpi=150, sharex=True, gridspec_kw={'height_ratios': [3.5, 1]})

# 1. ASPI Price and Moving Averages
ax1.plot(df_focus['Date_dt'], df_focus['ASPI'], color='#059669', linewidth=1.8, label='ASPI Index Close')
ax1.plot(df_focus['Date_dt'], df_focus['MA_20'], color='#0284c7', linewidth=1.2, linestyle='--', alpha=0.8, label='20-Day MA')
ax1.plot(df_focus['Date_dt'], df_focus['MA_50'], color='#9333ea', linewidth=1.2, linestyle=':', alpha=0.8, label='50-Day MA')

# 2. Overlay Policy Events
colors = {'Fiscal Policy': '#f59e0b', 'Emergency Economic Policy': '#ef4444', 'Tax Policy': '#8b5cf6', 'Foreign Exchange Policy': '#ec4899'}
for _, pol in policies_focus.iterrows():
    p_date = pol['Aligned_Date_dt']
    p_close = pol['ASPI_Event_Close']
    c = colors.get(pol['Policy_Type'], '#eab308')
    ax1.scatter(p_date, p_close, color=c, s=55, zorder=5, edgecolor='black', linewidth=0.8)

ax1.set_title('Colombo Stock Exchange — ASPI 2020–2025 Timeline with Government Policy Interventions', fontsize=12, fontweight='bold')
ax1.set_ylabel('Index Level (LKR)')
ax1.grid(True, linestyle='--', alpha=0.4)
ax1.legend(loc='upper left', framealpha=0.9)

# 3. Turnover Volume Subplot
ax2.bar(df_focus['Date_dt'], df_focus['Turnover_Equity_Mn'], color='#0284c7', alpha=0.6, width=1.0)
ax2.set_ylabel('Turnover (Mn)')
ax2.set_xlabel('Timeline (2020–2025)')
ax2.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
fig1_path = os.path.join(REPORTS_DIR, 'aspi_2020_2025_with_policies.png')
plt.savefig(fig1_path)
plt.show()
print(f"Figure 1 saved to: {fig1_path}")

### 6. High-Resolution Figure 2: Policy Impact Across Return Horizons
Compare the empirical distribution of market returns across 1-day, 3-day, and 5-day post-announcement horizons.

In [ ]:
plt.figure(figsize=(12, 5), dpi=150)

ret_data = []
for horizon in ['Return_Post_1d', 'Return_Post_3d', 'Return_Post_5d']:
    vals = df_aligned[horizon].dropna() * 100
    for v in vals:
        ret_data.append({'Horizon': horizon.replace('Return_Post_', '').replace('d', ' Trading Days'), 'Cumulative Return (%)': v})

df_ret = pd.DataFrame(ret_data)
sns.boxplot(x='Horizon', y='Cumulative Return (%)', data=df_ret, palette=['#10b981', '#38bdf8', '#a855f7'], width=0.4)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.title('Distribution of CSE ASPI Returns Across Post-Policy Horizons (2000–2025)', fontsize=11, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
fig2_path = os.path.join(REPORTS_DIR, 'policy_impact_distribution_horizons.png')
plt.savefig(fig2_path)
plt.show()
print(f"Figure 2 saved to: {fig2_path}")

### 7. Export Descriptive Statistics and Timeline Metadata
Generate summary statistics JSON for research documentation.

In [ ]:
stats = {
    "timeline_scope": "2020-2025 Primary Evaluation Horizon",
    "total_trading_days": int(len(df_focus)),
    "starting_close": float(df_focus['ASPI'].iloc[0]),
    "ending_close": float(df_focus['ASPI'].iloc[-1]),
    "period_net_return_pct": round(float(((df_focus['ASPI'].iloc[-1] - df_focus['ASPI'].iloc[0]) / df_focus['ASPI'].iloc[0]) * 100), 2),
    "period_high": float(df_focus['ASPI'].max()),
    "period_low": float(df_focus['ASPI'].min()),
    "annualized_volatility": round(float(df_focus['ASPI'].pct_change().std() * np.sqrt(252) * 100), 2),
    "policy_count_focus": int(len(policies_focus)),
    "mean_immediate_1d_reaction_pct": round(float(df_aligned['Return_Post_1d'].dropna().mean() * 100), 2),
    "mean_3d_reaction_pct": round(float(df_aligned['Return_Post_3d'].dropna().mean() * 100), 2),
    "mean_5d_reaction_pct": round(float(df_aligned['Return_Post_5d'].dropna().mean() * 100), 2)
}

out_json = os.path.join(REPORTS_DIR, 'timeline_descriptive_stats.json')
with open(out_json, 'w') as f:
    json.dump(stats, f, indent=2)
print("Descriptive statistics exported:", out_json)
print(json.dumps(stats, indent=2))